# TripAdvisor Sentiment Analysis — Step 1: Web Scraping

Welcome to the first stage of your research project! In this notebook, we will extract raw hotel guest reviews directly from TripAdvisor.

### Learning Objectives:
1. Understand the concept of **Web Scraping**.
2. Learn how to extract HTML elements (titles, texts, ratings) using `rvest`.
3. Handle anti-bot protections and implement fallback datasets.

---

### ☁️ Google Colab Setup
If you are running this notebook in **Google Colab**, simply run the code cell below to download the dataset and install any missing tools. If you are running this locally in RStudio, you can skip it!

In [ ]:
# =====================================================================
# GOOGLE COLAB SETUP (Run this cell only if you are using Google Colab)
# =====================================================================
if (dir.exists("/content")) {
  if (!dir.exists("/content/MrKadekProject")) {
    cat("Downloading project files from GitHub...\n")
    system("git clone https://github.com/divanahadyan1618/MrKadekProject.git /content/MrKadekProject", ignore.stdout=TRUE, ignore.stderr=TRUE)
  }
  setwd("/content/MrKadekProject")
  
  # Install required packages if missing
  packages <- c("rvest", "tidytext", "stopwords", "syuzhet", "wordcloud", "RColorBrewer")
  new_packages <- packages[!(packages %in% installed.packages()[,"Package"])]
  if(length(new_packages)) install.packages(new_packages, quiet = TRUE)
  
  cat("Colab Environment Ready!\n")
}


## 1. Load Required Packages

We need several libraries to read web pages and manipulate the downloaded data.

In [ ]:
# =====================================================================
# TripAdvisor Sentiment Analysis - Step 1: Web Scraping
# =====================================================================
# Welcome! If you have never programmed before, do not worry.
# We will explain exactly what the computer is doing at every single step.
#
# PURPOSE OF THIS SCRIPT:
# We want to go to TripAdvisor, find a specific hotel, and automatically
# copy-paste hundreds of reviews into an Excel-like table so we can analyze them.
# This automatic copying is called "Web Scraping".

## 2. Define Target URL & Scraper Function

We define the TripAdvisor URL we want to scrape. We will build a custom function `scrape_tripadvisor_reviews()` that identifies specific **CSS classes** (like `span.yUiMA` or `span.ui_bubble_rating`) to pull exactly the text we want.

In [ ]:
# =====================================================================
# STEP 1: Load Required Packages (Adding New Tools to R)
# =====================================================================
# Think of R as a blank toolbox. "Packages" are specialized toolsets created
# by other programmers that we borrow to make our job easier.
# The 'library()' command tells the computer to open and equip that toolset.

# The 'rvest' toolset helps us download and read web pages.
library(rvest)
# The 'tidyverse' toolset helps us sort, filter, and organize data.
library(tidyverse)
# The 'xml2' toolset safely reads the raw computer code of a website.
library(xml2)

# 'cat()' is a command that simply prints a message on your screen.
cat("Scraping packages loaded successfully!\n")

## 3. Execute Scraper & Handle Fallback Data

Because TripAdvisor frequently blocks bots using Cloudflare, we execute our scraper inside a conditional block. If it fails to scrape live data, it will automatically initialize a realistic, 1000-row synthetic dataset so that our research can continue seamlessly.

In [ ]:
# =====================================================================
# STEP 2: Define Where We Want to Go
# =====================================================================
# The '<-' arrow is called the "assignment operator". 
# It means "take the internet link on the right, and save it inside a box named 'hotel_url' on the left."
hotel_url <- "https://www.tripadvisor.com/Hotel_Review-g297698-d607449-Reviews-Bvlgari_Resort_Bali-Uluwatu_Bukit_Peninsula_Bali.html"

## 4. Export Raw Data

Finally, we save the downloaded/generated data locally. We use `write_excel_csv2` to output a semicolon-delimited CSV, ensuring it opens perfectly in Indonesian and European versions of Microsoft Excel.

In [ ]:
# =====================================================================
# STEP 3: Create the "Scraper" Machine
# =====================================================================
# Here we are building our own custom tool (a "function") named 'scrape_tripadvisor_reviews'.
# Think of it like a recipe: we give it a web link (url), and it gives us a table of reviews back.
scrape_tripadvisor_reviews <- function(url) {
  
  # 'tryCatch' tells the computer: "Try to do this, but if the website blocks you, don't crash!"
  tryCatch({
    # 1. Download the webpage. 'user_agent' disguises our computer so TripAdvisor thinks we are a normal Chrome browser.
    page <- read_html(x = url, options = "RECOVER", 
                      user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    # The '%>%' symbol means "AND THEN". 
    # Example: Take the page AND THEN find the text AND THEN clean it up.
    
    # 2. Extract the Review Text by looking for TripAdvisor's specific font/design codes ("span.yUiMA")
    reviews <- page %>% html_nodes("span.yUiMA, div.fIrGe") %>% html_text(trim = TRUE)
    
    # 3. Extract the Review Titles (the bold headline of the review)
    titles <- page %>% html_nodes("span.yUiMA, a.QYdug") %>% html_text(trim = TRUE)
    
    # 4. Extract the Star Ratings ( TripAdvisor stores 5 stars as "50", so we divide by 10 )
    ratings <- page %>% html_nodes("span.ui_bubble_rating") %>% html_attr("class") %>% 
      str_extract("[0-9]+") %>% as.numeric() / 10
    
    # 5. Extract the Date the guest stayed at the hotel
    dates <- page %>% html_nodes("span.teSgY") %>% html_text(trim = TRUE) %>% 
      str_remove("Date of stay: ")
    
    # 6. 'tibble' means "Table". We are taking the 4 lists above and taping them together into a spreadsheet.
    df <- tibble(
      title = head(titles, length(reviews)),
      review_text = reviews,
      rating = head(ratings, length(reviews)),
      review_date = head(dates, length(reviews))
    )
    
    # Return the finished table back to us
    return(df)
    
  }, error = function(e) {
    # If the website blocks us, print a warning message instead of crashing.
    message("Warning: Live scraping was blocked or interrupted: ", e$message)
    return(NULL)
  })
}

## 🎓 Student Exercise / Assignment Connection

For your project assignment, make sure that:
1. You identify the correct `hotel_url` for the exact hotel you want to analyze.
2. Be aware that the CSS classes `span.yUiMA` might change over time as TripAdvisor updates its website design.

**Great job! Open `02_cleaning.ipynb` to clean the raw data.**

# TripAdvisor Sentiment Analysis — Step 2: Data Cleaning & Preprocessing

Welcome to the second stage of your research project! In this notebook, we will clean and preprocess the raw text data collected in Step 1.

### Learning Objectives:
1. Explain why text data cleaning (preprocessing) is critical in text mining.
2. Remove symbols, punctuation, HTML tags, numbers, emojis, and stopwords.
3. Perform **Tokenization** to break sentences into words.
4. Save clean datasets for sentiment scoring.

---

## 1. The Importance of Data Cleaning

Raw e-WOM reviews are noisy. They contain emojis, punctuation like `!!!`, extra spaces, and common functional words like *"the"*, *"is"*, and *"at"* (stopwords) that do not carry sentimental meaning. 

If we don't clean this noise, the computer will struggle to identify true sentiment patterns.

**Example:**
* *Original sentence:* `"The hotel was sooo gooood!!! 😍"`
* *After cleaning:* `"hotel good"`

Let's load the required packages (`tidyverse` and `tidytext`) and the custom R helper scripts (`helpers.R`) which contain our specific data-cleaning algorithms.

In [ ]:
# =====================================================================
# TripAdvisor Sentiment Analysis - Step 2: Data Cleaning & Preprocessing
# =====================================================================
# Welcome! If you have never programmed before, do not worry.
# We will explain exactly what the computer is doing at every single step.
#
# PURPOSE OF THIS SCRIPT:
# Text written by humans is very "messy". We use emojis, punctuation like "!!!",
# and slang like "u" instead of "you". Before a computer can understand emotions, 
# we have to scrub all this "noise" away. This is called Text Pre-Processing.

## 2. Load Raw Reviews Data

We read the raw TripAdvisor data saved from Step 1. We use `read_csv2()` to guarantee perfect column alignment for semicolon-delimited files, which is the standard format for systems configured in European and Indonesian regions.

In [ ]:
# =====================================================================
# STEP 1: Load Required Packages (Adding Tools)
# =====================================================================
# 'tidyverse' helps us filter and change table data easily.
library(tidyverse)
# 'tidytext' is a specialized toolset used purely for analyzing text and words.
library(tidytext)

# We wrote a file called 'helpers.R' that contains our secret cleaning recipes.
# 'source()' tells the computer to load our custom recipes into memory.
source("scripts/helpers.R")

cat("Helper functions and libraries loaded!\n")

## 3. Apply Text Cleaning Pipeline

We apply our custom text cleaning functions to the review text column:
1. `clean_text()`: removes HTML, numbers, extra spaces, punctuations, and lowercase the text.
2. `normalize_slang()`: converts common slang (e.g. `sooo good`, `u`) to standard dictionary words.

We then print a comparison to see how dramatically the text has been simplified.

In [ ]:
# =====================================================================
# STEP 2: Open the Raw Data
# =====================================================================
# 'read_csv2' reads the Excel file we created in Step 1 and puts it in a 
# box named 'raw_data' so we can look at it.
raw_data <- read_csv2("data/raw/bvlgari_raw_reviews.csv")

cat("Successfully loaded", nrow(raw_data), "raw reviews!\n")

## 4. Tokenization & Stopwords Removal

**Tokenization** breaks entire review sentences into individual units (usually single words, called **unigrams**). We use the `tidytext` package function `unnest_tokens()` to do this. The resulting dataframe will have 1 row per word, increasing the row count heavily.

Then, we remove standard English **stopwords** (e.g., *"the"*, *"and"*, *"is"*, *"at"*) using our helper function `remove_stopwords()`. These words occur very frequently but carry absolutely zero sentimental meaning. Removing them optimizes the dataset for the sentiment algorithms.

In [ ]:
# =====================================================================
# STEP 3: The Cleaning Pipeline
# =====================================================================
# The %>% symbol is a "pipe". It means "AND THEN".
# We are taking the raw data, AND THEN changing (mutating) the text column.
# 
# Our custom 'clean_text' recipe removes numbers, exclamation marks, and emojis.
# Our custom 'normalize_slang' recipe fixes internet slang (e.g., sooo -> so).

cleaned_reviews_df <- raw_data %>%
  mutate(
    # Create a new column called 'cleaned_text' that contains the scrubbed reviews.
    cleaned_text = clean_text(review_text),
    cleaned_text = normalize_slang(cleaned_text)
  )

# Let's print out the 2nd row to see what the computer did!
cat("--- THIS WAS THE MESSY ORIGINAL TEXT ---\n", cleaned_reviews_df$review_text[2], "\n\n")
cat("--- THIS IS THE PERFECTLY CLEANED TEXT ---\n", cleaned_reviews_df$cleaned_text[2], "\n")

## 5. Export Cleaned Datasets

We will save two datasets in `data/cleaned/`:
1. `bvlgari_cleaned_reviews.csv` — Full reviews with a column for cleaned review sentences (used for sentence scoring).
2. `bvlgari_cleaned_tokens.csv` — Tokenized reviews (one row per word) for word-frequency analysis and wordclouds.

We use `write_excel_csv2` to output semicolon-delimited CSVs for perfect formatting compatibility.

In [ ]:
# =====================================================================
# STEP 4: Tokenization (Chopping Sentences into Words)
# =====================================================================
# Computers don't read paragraphs; they read individual words.
# 'unnest_tokens' is a tool that takes a full sentence and chops it up.
# If a review is 10 words long, this tool creates 10 separate rows in our table!
# This process is formally called "Tokenization".

tokenized_df <- cleaned_reviews_df %>%
  unnest_tokens(output = word, input = cleaned_text)

cat("After chopping the sentences, we have", nrow(tokenized_df), "individual words!\n")

## 🎓 Student Exercise / Assignment Connection

For your project assignment, make sure that:
1. You understand what a stopword is and why it should be deleted. 
2. If you are scraping an Indonesian hotel/destination, you must replace the English stopword lexicons with an Indonesian stopword list (using the `stopwords` package in R with source = `"stopwords-iso"` and language = `"id"`).

**Great job! Open `03_sentiment_analysis.ipynb` to calculate sentiment scores.**

# TripAdvisor Sentiment Analysis — Step 3: Sentiment Lexicon Scoring

Welcome to the third stage of your research project! In this notebook, we will apply **Lexicon Models** to identify emotional meaning in words and calculate sentiment scores for each guest review.

### Learning Objectives:
1. Understand what a **Sentiment Lexicon** is.
2. Compare different lexicons (**AFINN**, **NRC**, and **Syuzhet**).
3. Compute word-level and review-level sentiment scores.
4. Classify reviews into Positive, Negative, and Neutral groups.

---

## 1. Sentiment Lexicon Scoring Concept

A **Sentiment Lexicon** is a dictionary of words tagged with emotional meaning or numerical polarity scores.

**Examples:**
* `"excellent"` ➔ Positive (+4 in AFINN)
* `"dirty"` ➔ Negative (-3 in AFINN)
* `"slow"` ➔ Negative (-2 in AFINN)

The total sentiment score of a review is calculated by summing the scores of all its sentiment-carrying words.

Let's load the libraries. We will use the `syuzhet` package, which is the gold-standard in R for computing sentiment scores using multiple dictionary methods easily.

In [ ]:
# =====================================================================
# TripAdvisor Sentiment Analysis - Step 3: Sentiment Lexicon Scoring
# =====================================================================
# Welcome! If you have never programmed before, do not worry.
# We will explain exactly what the computer is doing at every single step.
#
# PURPOSE OF THIS SCRIPT:
# How does a computer know if a review is happy or angry? It uses a "Lexicon".
# A Lexicon is simply a giant dictionary where scientists have manually assigned 
# numeric scores to words. For example: "excellent" = +4, "dirty" = -3.
# We will scan our clean reviews against these dictionaries to calculate a final score.

## 2. Load Cleaned Data

We read the cleaned full-sentence reviews dataframe from Step 2 to evaluate the context of each review.

In [ ]:
# =====================================================================
# STEP 1: Load Required Packages
# =====================================================================
# 'tidyverse' and 'tidytext' help us read and manipulate our tables.
library(tidyverse)
library(tidytext)

# 'syuzhet' is a very famous toolset built specifically for Sentiment Analysis.
# It contains the massive dictionaries we need to score emotions.
library(syuzhet)

cat("Libraries loaded successfully!\n")

## 3. Scoring Sentiment using the Syuzhet Package

The `syuzhet` package provides simple functions to score text using several lexicons:
* **Syuzhet Lexicon**: Developed by the NLP laboratory at Nebraska-Lincoln (default NLP method).
* **AFINN Lexicon**: Scores words on an integer scale from -5 (very negative) to +5 (very positive).

Let's calculate numeric sentiment scores for each review using these methods.

In [ ]:
# =====================================================================
# STEP 2: Open the Cleaned Reviews
# =====================================================================
# We load the full cleaned sentences from Step 2 into a box named 'cleaned_reviews'.
cleaned_reviews <- read_csv2("data/cleaned/bvlgari_cleaned_reviews.csv")

cat("Loaded", nrow(cleaned_reviews), "cleaned reviews for emotional scoring.\n")

## 4. Classify Sentiment Categories

We logically bucket the numeric `score_syuzhet` value into distinct nominal categories:
* **Positive**: score > 0
* **Neutral**: score == 0
* **Negative**: score < 0

We then print out the percentage distribution to see how guests feel overall.

In [ ]:
# =====================================================================
# STEP 3: Apply the Dictionary Lexicons (Syuzhet & AFINN)
# =====================================================================
# We are asking the computer to read every review and calculate a math score.
# We use two different methods to be thorough:
# 1. "syuzhet" method: A standard scientific scoring system.
# 2. "afinn" method: Gives an integer score from -5 (furious) to +5 (thrilled).

# 'mutate' means "create a new column in our table".
sentiment_results <- cleaned_reviews %>%
  mutate(
    score_syuzhet = get_sentiment(cleaned_text, method = "syuzhet"),
    score_afinn = get_sentiment(cleaned_text, method = "afinn")
  )

## 5. Rich Emotion Extraction (NRC Lexicon)

The **NRC Emotion Lexicon** is a list of words that associates text with eight basic human emotions: anger, anticipation, disgust, fear, joy, sadness, surprise, and trust.

**Understanding NRC Scores:** The numerical scores produced by the NRC lexicon represent the *frequency* or *count* of words associated with each specific emotion category found in the text. For example, a joy score of 3 means there were 3 words in the review associated with the emotion of joy.

Let's extract detailed emotion scores for Bvlgari Resort Bali reviews. *Note: This takes a little longer to execute as it checks against a massive dictionary.*

In [ ]:
# =====================================================================
# STEP 4: Classify the Scores into Simple Categories
# =====================================================================
# A numeric score like "1.45" is hard to understand.
# We use 'case_when' (which is like a giant IF statement) to categorize them:
# IF the score is greater than 0  -> "Positive"
# IF the score is less than 0     -> "Negative"
# IF the score is exactly 0       -> "Neutral"

sentiment_classified <- sentiment_results %>%
  mutate(
    sentiment_category = case_when(
      score_syuzhet > 0 ~ "Positive",
      score_syuzhet < 0 ~ "Negative",
      TRUE              ~ "Neutral" # TRUE here means "everything else"
    )
  )

# Now, let's group all the "Positive" and "Negative" rows together and count them!
sentiment_summary <- sentiment_classified %>%
  group_by(sentiment_category) %>%
  summarise(
    count = n(), # n() counts the number of rows
    percentage = (n() / nrow(sentiment_classified)) * 100 # Calculate the percentage %
  )

# Print the final report to the screen
print(sentiment_summary)

## 6. Save Scored Data

We save our final, heavily annotated dataset to `data/cleaned/bvlgari_sentiment_scores.csv` so it is ready for Step 4 (Data Visualization).

In [ ]:
# =====================================================================
# STEP 5: NRC Deep Emotion Extraction
# =====================================================================
# "Positive" or "Negative" is too basic. What if we want to know if guests are ANGRY or AFRAID?
# The 'NRC' lexicon checks text against 8 human emotions:
# (anger, anticipation, disgust, fear, joy, sadness, surprise, trust).

cat("Processing deep emotions. The computer is reading thousands of words, please wait a few seconds...\n")

# 'get_nrc_sentiment' returns a table showing exactly how much 'joy' or 'anger' is in the text.
nrc_emotions <- get_nrc_sentiment(sentiment_classified$cleaned_text)

# 'bind_cols' acts like glue. It takes our original table, and glues the new emotion columns to the right side of it.
final_research_data <- bind_cols(sentiment_classified, nrc_emotions)

## 🎓 Student Exercise / Assignment Connection

For your project assignment, make sure that:
1. You understand the difference between **sentiment score** (numeric scale, e.g. -2, 4) and **emotion category** (nominal category, e.g. joy, trust).
2. If you are comparing two different hotels, you should run this scoring notebook for both hotels, then compare their average scores to determine which hotel has superior customer sentiments.

**Great job! Let's open `04_visualization.ipynb` to create beautiful visuals.**

# TripAdvisor Sentiment Analysis — Step 4: Data Visualization & Insights

Welcome to the final stage of your research project! In this notebook, we transform the complex numeric sentiment and emotion data into beautiful, publish-ready visualizations.

### Learning Objectives:
1. Learn how to map sentiment categories (Positive/Negative) into a Bar Chart.
2. Visualize deep emotional profiles (Joy, Trust, Anger) using horizontal columns.
3. Build a beautiful **Word Cloud** to identify commonly used terms.
4. Plot a timeline trend of guest sentiment to track operational quality over time.

---

## 1. Load Packages and Data

We load `ggplot2` (part of the `tidyverse`) to create high-quality graphics, and `wordcloud` to generate frequency clouds.

In [ ]:
# =====================================================================
# TripAdvisor Sentiment Analysis - Step 4: Data Visualization & Insights
# =====================================================================
# Welcome! If you have never programmed before, do not worry.
# We will explain exactly what the computer is doing at every single step.
#
# PURPOSE OF THIS SCRIPT:
# Math numbers and huge spreadsheets are boring and hard to read.
# "Data Visualization" means turning those numbers into beautiful, colorful 
# pictures (like bar charts and line graphs) so we can easily understand the story!

## 2. Define Custom ggplot2 Aesthetic Theme

We define a custom premium theme so all our charts have a cohesive, professional, academic design without needing to copy-paste the formatting logic for every plot.

In [ ]:
# =====================================================================
# STEP 1: Load Packages and Data
# =====================================================================
# We load our toolsets again. 
library(tidyverse)    # Contains 'ggplot2' - the most powerful charting tool in R.
library(tidytext)     # For handling our text data.
library(wordcloud)    # A fun tool specifically for drawing Word Clouds.
library(RColorBrewer) # A tool that provides beautiful, professional color palettes.

# We load the final, scored dataset from Step 3.
research_data <- read_csv2("data/cleaned/bvlgari_sentiment_scores.csv")
# We also load the chopped up individual words from Step 2 (for the word cloud).
cleaned_tokens <- read_csv2("data/cleaned/bvlgari_cleaned_tokens.csv")

cat("Data and drawing tools loaded successfully!\n")

## 3. Plot Sentiment Class Distribution

We create a standard bar chart comparing the total volume of Positive vs Negative vs Neutral reviews, assigning semantic colors to each.

In [ ]:
# =====================================================================
# STEP 2: Create a Custom "Theme" (Paintbrush)
# =====================================================================
# Instead of telling the computer "make the title bold and size 16" every 
# single time we draw a chart, we create a 'theme_premium' recipe once.
# Now, we just slap 'theme_premium()' on any chart, and it instantly looks professional!

theme_premium <- function() {
  theme_minimal() + # Start with a clean, white background
  theme(
    text = element_text(color = "#2C3E50"), # Dark blue/grey text
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5, color = "#1A252C"), # Centered bold title
    plot.subtitle = element_text(size = 11, hjust = 0.5, color = "#5A6F7C"), # Centered subtitle
    axis.title = element_text(face = "bold", size = 11), # Bold axis labels (X and Y)
    legend.position = "none", # Hide the messy legend box
    panel.grid.minor = element_blank() # Remove distracting background grid lines
  )
}

## 4. Plot NRC Emotional Profile

We sum up the specific emotional values (anger, joy, trust, etc.) across the entire dataset to see what the dominant emotional experience is like at the hotel.

In [ ]:
# =====================================================================
# STEP 3: Draw the Sentiment Distribution Chart (Positive vs Negative)
# =====================================================================
# We use 'ggplot' to draw the chart. 
# Think of 'aes' (aesthetics) as telling the computer where to put the data.
# x = sentiment_category means "Put Positive, Neutral, and Negative on the bottom X axis".
# fill = sentiment_category means "Color them based on their category".

p1 <- ggplot(research_data, aes(x = sentiment_category, fill = sentiment_category)) +
  geom_bar(width = 0.6, alpha = 0.85) + # Draw solid bars (alpha makes them slightly transparent)
  # scale_fill_manual lets us pick the exact paint colors. Green = Good, Red = Bad, Grey = Neutral.
  scale_fill_manual(values = c("Positive" = "#16A085", "Negative" = "#E74C3C", "Neutral" = "#95A5A6")) +
  # 'labs' stands for labels. We give our chart a title.
  labs(
    title = "Sentiment Distribution of TripAdvisor Reviews",
    subtitle = "Bvlgari Resort Bali guest opinions classification",
    x = "Sentiment Category",
    y = "Number of Reviews"
  ) +
  theme_premium() # Apply our custom paintbrush from Step 2!

# 'ggsave' saves the picture to our computer folder as a PNG image file.
ggsave("output/figures/sentiment_distribution.png", plot = p1, width = 7, height = 5, dpi = 300)

## 5. Generate the Word Cloud

Word clouds are beautiful ways to see the most frequently used words. The larger the word, the more frequently it was mentioned by guests. We calculate word frequencies, excluding non-descriptive filler context words (e.g. 'hotel', 'bali' are universally mentioned and waste visual space).

In [ ]:
# =====================================================================
# STEP 4: Draw the Deep Emotion Chart (Joy, Trust, Anger, etc.)
# =====================================================================
# We calculate the total sum of all the 8 emotions.
emotions_summary <- research_data %>%
  select(anger, anticipation, disgust, fear, joy, sadness, surprise, trust) %>%
  summarise(across(everything(), sum)) %>% # Add them all up
  pivot_longer(cols = everything(), names_to = "emotion", values_to = "score") %>% # Reshape the table so we can graph it
  arrange(desc(score)) # Sort them from highest to lowest

# Draw the chart!
# 'reorder' automatically sorts the bars so the tallest one is at the top.
p2 <- ggplot(emotions_summary, aes(x = reorder(emotion, score), y = score, fill = emotion)) +
  geom_col(alpha = 0.85, width = 0.7) + # Draw the columns
  coord_flip() + # Flip the chart sideways! Horizontal charts are easier to read.
  scale_fill_brewer(palette = "Dark2") + # Use a built-in professional color palette
  labs(
    title = "Detailed Emotional Profile of Guest Experience",
    subtitle = "NRC Lexicon emotional analysis metrics for Bvlgari Resort Bali",
    x = "Emotion Dimension",
    y = "Total Emotion Score"
  ) +
  theme_premium()

ggsave("output/figures/emotions_breakdown.png", plot = p2, width = 8, height = 5, dpi = 300)

## 6. Plot Timeline Sentiment Trend

Line charts are excellent for tracking operational changes over time. By tracking average sentiment scores over time (grouped by month), researchers can pinpoint when a hotel's service declined or improved.

In [ ]:
# =====================================================================
# STEP 5: Draw the Word Cloud
# =====================================================================
# A Word Cloud shows the most frequently used words. The bigger the word, the more it was used!
# We 'filter' out boring words like 'hotel' and 'bali' because they waste space.
word_counts <- cleaned_tokens %>%
  count(word, sort = TRUE) %>%
  filter(!word %in% c("hotel", "resort", "bali", "stay", "room", "villa", "service"))

cat("Drawing the word cloud...\n")

# Open a digital "canvas" to draw the picture on
png("output/figures/wordcloud.png", width = 800, height = 800, res = 150)
set.seed(123) # Lock the randomness so the cloud looks exactly the same every time we run it

# Call the wordcloud drawing tool
wordcloud(
  words = word_counts$word,   # What words to draw
  freq = word_counts$n,       # How big to draw them based on frequency
  min.freq = 2,               # Ignore words that only appeared once
  max.words = 40,             # Stop drawing after 40 words so it doesn't look messy
  random.order = FALSE,       # Put the biggest words right in the center
  rot.per = 0.2,              # Randomly rotate 20% of the words sideways
  colors = brewer.pal(8, "Dark2") # Paint them with cool colors
)
dev.off() # Close the digital canvas and save the file!

## 🎓 Conclusion

You have successfully completely a full e-WOM Sentiment Analysis pipeline! You learned how to scrape data, clean text, apply lexicons, and visualize emotional patterns. You can use these exact notebooks to process any other hotel or brand on TripAdvisor for your research.